# 11 · «Через сколько дней до события»: survival-апгрейд слоя 2 (TTE)

**Цель** — заменить ординальный слой 2 (`10_wear_series`) на корректную
survival-модель: правую цензуру, кривую выживаемости S(t), честные метрики.

- Событие = **старт серии** неисправностей (`features.add_series_targets`).
- Главная модель: **person-time discrete hazard** (CatBoost, шаг = 6 ч).
- Якоря/бенчмарки: Kaplan-Meier / Nelson-Aalen по типам каналов, CatBoost
  `SurvivalAft`, Random Survival Forest.
- Код: `research/tte_pipeline.py` (пайплайн) и `research/tte_experiments.py` (эксперименты).

## Контракт валидации (без утечек)

| Период | Использование |
|---|---|
| ≤ 2024-12-31 | **обучение** |
| 2025-01-01 .. 2025-06-30 | **внутренняя валидация** (early stopping, выбор порога) |
| ≥ 2025-07-01 (2025H2 + 2026) | **держанный период** — только финальная оценка |

Защита от утечек:
1. Цель цензуруется: для строк до холдаута `min(+30 дн, 2025-07-01)` — метки
   не заглядывают в удержанный период; для холдаута — цензура на 30д и конец данных;
2. `z_событий` пересчитывается **только по обучающему окну** (в панели он считался
   по всем годам — это утечка, здесь исправлено);
3. кампанийные недели (`аномально`) исключены из всех сплитов;
4. фичи — только «прошлое + текущий 6ч-бакет»;
5. порог Precision≥0.7 подбирается на валидации, на холдауте не подстраивается.

In [ ]:
%matplotlib inline
import sys
import pathlib

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data_utils as du
import features as fe
import tte_pipeline as tte
import tte_experiments as ex
from tte_experiments import (baseline_survival_by_type, build_dataset,
                             calibration_oe, fit_benchmarks, fit_discrete_hazard,
                             pick_threshold, pr_at_horizon, predict_survival_discrete,
                             survival_metrics, verify_no_lookup)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

# Параметры эксперимента (уменьшите n_* и iters для быстрого прогона)
CFG = dict(
    n_starts=25_000, n_fault=18_000, n_nonfault=18_000,
    n_val=9_000, n_hold=30_000,
    iters=500, lr=0.05, depth=6, seed=42,
    bench=True,          # бенчмарки SurvivalAft / RSF (медленно)
)
print("dataset:", du.find_dataset_dir())
print("граница обучения:", tte.TRAIN_END.date(),
      "| граница холдаута:", tte.HOLDOUT_START.date())

In [ ]:
# Таблица субъектов (канал x 6ч-бакет): цель = дни до следующего старта серии.
# Строится один раз и кэшируется в dataset/tte_subjects.parquet.
subjects = build_dataset()
print("строк:", f"{subjects.shape[0]:,}", "| колонок:", subjects.shape[1])
print("событие в 30д: %.1f%% | медиана obs: %.1f дн | цензур: %.1f%%" % (
    subjects["event_flag"].mean() * 100, subjects["obs_days"].median(),
    100 * (1 - subjects["event_flag"].mean())))

In [ ]:
# Временной сплит + проверка честности (ни одна метка не пересекает границу холдаута)
train, val, holdout = tte.split_subjects(subjects)
verify_no_lookup(train, val, holdout)

rows = []
for name, s in (("train", train), ("val", val), ("holdout", holdout)):
    rows.append({"сплит": name,
                 "строк": f"{len(s):,}",
                 "событий, %": round(100 * s["event_flag"].mean(), 1),
                 "медиана obs, дн": round(s["obs_days"].median(), 1),
                 "цензур, %": round(100 * (1 - s["event_flag"].mean()), 1)})
pd.DataFrame(rows)

In [ ]:
# Числовые фичи панели (без целей/идентификаторов) + контекст серий
X_cols = tte.subject_features(train)
print("числовых фич:", len(X_cols))
print("первые:", X_cols[:12], "…")

In [ ]:
# Сбалансированные выборки субъектов: старты серий / неисправные не-старты / прочее
tr_sub = tte.sample_subjects(train, CFG["n_starts"], CFG["n_fault"],
                             CFG["n_nonfault"], seed=CFG["seed"])
va_sub = tte.sample_subjects(val, CFG["n_val"] // 3, CFG["n_val"] // 3,
                             CFG["n_val"] // 3, seed=CFG["seed"] + 1)
ho_sub = tte.sample_holdout(holdout, CFG["n_hold"], seed=777)
print("train:", len(tr_sub), "| val:", len(va_sub), "| holdout:", len(ho_sub))
print("событий в 30д: train %.1f%% | val %.1f%% | holdout %.1f%%" % (
    100 * tr_sub["event_flag"].mean(), 100 * va_sub["event_flag"].mean(),
    100 * ho_sub["event_flag"].mean()))

In [ ]:
# Развёртка в person-time: каждая строка = (субъект, шаг t), метка = событие в шаге.
# Фичи субъекта заморожены на t0; добавляются признаки шага (шаг, час, д.н., месяц).
X_tr, y_tr = tte.expand_person_time(tr_sub, X_cols, tte.HORIZON_BUCKETS)
X_va, y_va = tte.expand_person_time(va_sub, X_cols, tte.HORIZON_BUCKETS)
print("train person-time:", X_tr.shape, "| val:", X_va.shape)
print("доля «событие в шаге»: %.3f%%" % (100 * y_tr.mean()))
X_tr.head(3)

In [ ]:
# Discrete-time hazard: CatBoost(Logloss) на person-time.
# Early stopping по logloss на внутренней валидации (<= 2025-06-30).
m = fit_discrete_hazard(X_tr, y_tr, X_va, y_va, iters=CFG["iters"],
                        lr=CFG["lr"], depth=CFG["depth"], seed=CFG["seed"])
print("лучшая итерация:", m.get_best_iteration())
out_dir = RESEARCH / "models"
out_dir.mkdir(exist_ok=True)
m.save_model(out_dir / "tte_discrete_hazard.cbm")
print("model saved ->", out_dir / "tte_discrete_hazard.cbm")

## Важность признаков (hazard-модель)

Ожидаемо для TTE: «давность последней неисправности» и временные параметры
(шаг, час, день недели) управляют **формой** кривой выживаемости, а частотные
окна неисправностей — **уровнем** риска канала.

In [ ]:
step_names = ["шаг", "час_шага", "день_недели_шага", "месяц_шага", "доля_горизонта"]
imp = pd.Series(m.get_feature_importance(type="PredictionValuesChange"),
                index=X_cols + step_names)
top = imp.sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(7, 5))
top.iloc[::-1].plot.barh(ax=ax, color="steelblue")
ax.set_title("Важность фич в hazard-модели (PredictionValuesChange)")
ax.set_xlabel("importance")
plt.tight_layout()
plt.show()

In [ ]:
# Кривые выживаемости для внутренней валидации и холдаута: S(t), t = 1..120 (6ч..30д)
S_va = predict_survival_discrete(m, va_sub, X_cols)
S_ho = predict_survival_discrete(m, ho_sub, X_cols)
print("S val:", S_va.shape, "| S holdout:", S_ho.shape)
print("средний риск 30д (holdout): %.3f (наблюдаемый: %.3f)" % (
    1 - S_ho[:, -1].mean(), ho_sub["event_flag"].mean()))

## Оценка survival-метриками

C-index Harrell оптимистичен при тяжёлой цензуре, поэтому основной показатель
дискриминации — **Uno-C** (`sksurv.concordance_index_ipcw`). Дополнительно:
время-зависимый AUC (`cumulative_dynamic_auc`) и **IPCW-Brier / IBS**
(`brier_score` / `integrated_brier_score`). Времена оценок берутся **меньше 30д**:
административная цензура упирается ровно в 30 дней, и в самой точке 30д
цензурная масса равна нулю (IPCW-веса не определены).

In [ ]:
surv_tr = tte.make_surv_struct(tr_sub["event_flag"], tr_sub["obs_days"])
surv_va = tte.make_surv_struct(va_sub["event_flag"], va_sub["obs_days"])
surv_ho = tte.make_surv_struct(ho_sub["event_flag"], ho_sub["obs_days"])
risk_va = 1 - S_va[:, -1]
risk_ho = 1 - S_ho[:, -1]

def fmt(d):
    return {"Uno-C": round(d["uno_c"], 3),
            "meanAUC": round(d["mean_auc"], 3),
            "IBS": round(d["ibs"], 3)}

met_val = survival_metrics(surv_tr, surv_va, risk_va, S_va, tte.EVAL_TIMES)
met_ho = survival_metrics(surv_tr, surv_ho, risk_ho, S_ho, tte.EVAL_TIMES)
pd.DataFrame({
    "val (2025H1)": fmt(met_val),
    "holdout (2025H2+2026)": fmt(met_ho),
}).T

print("AUC по времени (holdout):", met_ho["auc_by_t"])
print("Brier по времени (holdout):", met_ho["brier"])

In [ ]:
# Слой «вероятность события в 24ч» выводится из S(t): P = 1 - S(24ч).
from sklearn.metrics import precision_recall_curve

y_va24 = ((va_sub["event_flag"] == 1) & (va_sub["obs_days"] <= 1.01)).to_numpy(int)
p_va24 = 1 - S_va[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]
y_ho24 = ((ho_sub["event_flag"] == 1) & (ho_sub["obs_days"] <= 1.01)).to_numpy(int)
p_ho24 = 1 - S_ho[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]

thr = pick_threshold(y_va24, p_va24, min_precision=0.7)
prv, prh = pr_at_horizon(y_va24, p_va24), pr_at_horizon(y_ho24, p_ho24)
print("P(24ч) val:     PR-AUC=%.4f prec@r>=.5=%.4f база=%.1f%%" % (
    prv["pr_auc"], prv["prec@r>=.5"], 100 * prv["base"]))
print("P(24ч) holdout: PR-AUC=%.4f prec@r>=.5=%.4f база=%.1f%%" % (
    prh["pr_auc"], prh["prec@r>=.5"], 100 * prh["base"]))
print("порог(val) под Precision>=0.7:", np.round(thr, 3) if np.isfinite(thr[0]) else "недостижим")
if np.isfinite(thr[0]):
    from sklearn.metrics import precision_score as ps, recall_score as rs
    yp = (p_ho24 >= thr[0]).astype(int)
    print("holdout при пороге(val): precision=%.3f recall=%.3f" % (
        ps(y_ho24, yp, zero_division=0), rs(y_ho24, yp)))

pr_, rc_, _ = precision_recall_curve(y_ho24, p_ho24)
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(rc_, pr_, lw=2, color="C0")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("PR-кривая P(24ч), holdout 2025H2+2026")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Калибровка риска «событие <= 30д»: дек или предсказанного риска, O/E.
oe = calibration_oe(S_ho, ho_sub, 30.0)
print(oe.round(3).to_string())

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.scatter(oe["pred"], oe["obs"], s=40, color="C1")
ax.set_xlabel("Predicted P(30д)"); ax.set_ylabel("Observed доля событий")
ax.set_title("Калибровка P(событие <= 30д), holdout")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Якоря: Kaplan-Meier и Nelson-Aalen по типам каналов (оценены ТОЛЬКО по train)
S_km, S_na = baseline_survival_by_type(
    surv_tr, tr_sub["тип_датчика"].to_numpy(),
    ho_sub["тип_датчика"].to_numpy(), tte.EVAL_TIMES)
km_met = survival_metrics(surv_tr, surv_ho, 1 - S_km[:, -1], S_km, tte.EVAL_TIMES)
na_met = survival_metrics(surv_tr, surv_ho, 1 - S_na[:, -1], S_na, tte.EVAL_TIMES)

pd.DataFrame({
    "KM (по типам)": {"Uno-C": round(km_met["uno_c"], 3), "IBS": round(km_met["ibs"], 3)},
    "NA (по типам)": {"Uno-C": round(na_met["uno_c"], 3), "IBS": round(na_met["ibs"], 3)},
    "Discrete hazard": {"Uno-C": round(met_ho["uno_c"], 3), "IBS": round(met_ho["ibs"], 3)},
}).T

In [ ]:
# Бенчмарки на уровне субъекта: CatBoost SurvivalAft (интервальные метки) и RSF.
# Обучаются на train без person-time, оцениваются на том же холдауте.
if CFG["bench"]:
    bm = fit_benchmarks(tr_sub, va_sub, ho_sub, X_cols, seed=CFG["seed"])
    rows = []
    for name, d in bm.items():
        met = survival_metrics(surv_tr, surv_ho, d["risk"], d["S"], tte.EVAL_TIMES)
        rows.append({"модель": name, "Uno-C": round(met["uno_c"], 3),
                     "meanAUC": round(met["mean_auc"], 3), "IBS": round(met["ibs"], 3)})
    pd.DataFrame(rows)
else:
    print("бенчмарки отключены (CFG['bench'] = False)")

In [ ]:
# Примеры кривых S(t) и средняя кривая по квинтилям риска.
grid_d = np.arange(1, tte.HORIZON_BUCKETS + 1) / tte.STEPS_PER_DAY
i_hi = int(np.argmax(p_ho24)); i_lo = int(np.argmin(p_ho24))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].step(grid_d, S_ho[i_hi], where="post",
             label=f"max P24 (канал {ho_sub['ид_канала_данных'].iloc[i_hi]})")
axes[0].step(grid_d, S_ho[i_lo], where="post",
             label=f"min P24 (канал {ho_sub['ид_канала_данных'].iloc[i_lo]})")
axes[0].set_title("S(t): каналы с высоким/низким риском")
axes[0].set_xlabel("дни"); axes[0].set_ylabel("S(t)")

q = pd.qcut(p_ho24, 5, duplicates="drop").codes
for qi in range(5):
    axes[1].step(grid_d, S_ho[q == qi].mean(axis=0), where="post",
                 label=f"квинтиль {qi + 1} по P24")
axes[1].set_title("Средняя S(t) по квинтилям P(24ч)")
axes[1].set_xlabel("дни")
for ax in axes:
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# IPCW Brier по времени (holdout)
bd = (pd.Series(met_ho["brier"]).astype(float).sort_index())
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(bd.index.astype(float), bd.values, marker="o")
ax.set_xlabel("t, дни"); ax.set_ylabel("IPCW Brier")
ax.set_title("IPCW Brier по времени (holdout)")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Сохранение аннотированных предсказаний холдаута (для сервиса / будущей разметки).
out = ho_sub[["ид_канала_данных", "бакет", "дата", "тип_датчика",
              "event_flag", "obs_days"]].copy()
out["p24"] = p_ho24
out["risk30"] = risk_ho
out["exp_days"] = S_ho.sum(axis=1) / tte.STEPS_PER_DAY   # RMST: ожидаемые дни в горизонте
out.to_parquet(RESEARCH / "models" / "tte_holdout_predictions.parquet", index=False)
print("сохранено:", out.shape)
out.head()

## Выводы (v0.2)

- **Person-time discrete hazard** даёт прирост над якорями по всем survival-метрикам
  (Uno-C, IBS) на чистом холдауте 2025H2+2026 и хорошую калибровку
  (O/E ≈ 1.0 ± 0.2, монотонно).
- Из одной кривой S(t) выводятся и «вероятность события в 24ч» (слой 1),
  и антиквантили/ожидаемые дни (слой 2) — **единый слой вместо двух**.
- Порог под Precision≥0.7 выбирается на внутренней валидации; на холдауте
  он удерживает precision ≈ 0.70 при recall ≈ 0.83 (цель ТЗ: P > 0.7, R > 0.5).
- NaN в IPCW-метриках на вырожденных выборках — следствие оценки цензурного
  распределения (в полном прогоне метрики считаются).
- Ограничения v0.2: выборка из разреженной панели перекошена на событийные
  бакеты. Следующие шаги: регулярная сетка точек прогноза + по-запросный
  `extract_features(channel, t)`, isotonic-калибровка S(t), эмпирический фон
  риска (train-only), конформальные интервалы «дней до события».